# Day 2 - Slicing, Dicing & Mutating in PySpark

This notebook covers data manipulation techniques in PySpark:
- Reading Parquet files
- Slicing (Selecting columns)
- Dicing (Filtering rows)
- Mutating (Adding/transforming columns)
- Practical exercise: Parsing log files

## 1. Create SparkSession

`SparkSession` is the unified entry point for Spark operations.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("spark_day2")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/03 16:56:16 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/03 16:56:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/biswa/practice/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/03 16:56:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Load Parquet File

Read the parquet data saved in Day 1. Parquet preserves schema and is efficient for analytics.

In [ ]:
parquet_path = r"../dataset/parquet_output"

df = spark.read.parquet(parquet_path)
df

## 3. Slicing (Selecting Columns)

### Method 1: Direct column selection
`select()` picks specific columns from the DataFrame.

In [ ]:
df_1 = df.select('ID', 'FirstName', 'Country')
df_1

### Method 2: Using col() with alias
- `col()` gives a column reference for transformations
- `alias()` renames columns on the fly

In [ ]:
from pyspark.sql.functions import col

df_2 = df.select(
    col('ID').alias('CustomerID'),
    col('FirstName'),
    col('Country')
)

df_2

## 4. Dicing (Filtering Rows)

`filter()` keeps rows matching the condition.
- Use `&` for AND, `|` for OR, `~` for NOT
- Always wrap conditions in parentheses due to operator precedence

In [ ]:
df_3 = df.filter(
    (col('Country') == 'USA' ) & (col('Score') > 500)
)
df_3.show()

## 5. Mutate (Adding/Transforming Columns)

### Conditional column with when/otherwise
- `withColumn()` adds a new column or replaces existing one
- `when()` works like if-else (similar to CASE WHEN in SQL)
- Chain multiple `.when()` for multiple conditions
- `.otherwise()` is the default/fallback case

In [ ]:
from pyspark.sql.functions import when

In [ ]:
df_4 = df.withColumn(
    "Performance",
    when(col("Score") >= 800, "Excellent")
    .when(col("Score") >= 500, "Average")
    .otherwise("Needs Improvement")
)
df_4.show()

## 6. Practical Exercise: Parsing Log Files

**Problem:** Parse a raw text log file into a clean DataFrame with 4 columns:
- `timestamp` (Proper Timestamp type)
- `log_level` (Only "ERROR", "INFO", or "WARN")
- `user_id` (Integer type, replacing null strings with 0)
- `message` (The final text sentence)

### Read raw text file

In [ ]:
txt_path = r"../dataset/server_log.txt"
txt = spark.read.text(txt_path)
txt.take(100)

### Parse using split() and concat()
- `split()` breaks a string by delimiter into an array
- `concat()` joins multiple strings/columns together
- `lit()` adds a literal string value
- Array indices start from 0

In [ ]:
from pyspark.sql.functions import split, concat, lit
txt = txt.withColumn('timestamp', concat(split('value', ' ')[0], lit(' '), split('value', ' ')[1]))\
        .withColumn('log_level', split('value', ' ')[2])\
        .withColumn('user_id', split('value', ' ')[3])\
        .withColumn('message', concat(split('value', ' ')[5], lit(' '), split('value', ' ')[6], lit(' '), split('value', ' ')[7], lit(' '), split('value', ' ')[8]))
txt = txt.select('timestamp', 'log_level', "user_id", "message")
txt.show()

## 7. Stop Spark Session

Always stop the session to release cluster resources.

In [ ]:
spark.stop()